# Story Prototype: End-To-End Company Analysis

This notebook is closer to the app story: investment outcome, benchmark comparison, risk journey, business quality, valuation, and CEO events as one supporting chapter.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-cache")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from analysis_engine.calculations import (
    calculate_debt_to_cash,
    calculate_event_return,
    calculate_gross_margin,
    calculate_max_drawdown,
    calculate_net_margin,
    calculate_pe_ratio,
    calculate_revenue_growth,
    classify_growth_rate,
    classify_margin,
    classify_pe_level,
    compare_investments,
)
from analysis_engine.data_access import annual_statement_rows, load_events, load_fundamentals, load_prices
from analysis_engine.utils import ceo_events, compact_money, money, pct, row_on_or_after, row_on_or_before

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)


## Inputs

The start date is early enough for AAPL to include the Steve Jobs to Tim Cook CEO transition.

In [ ]:
TICKER = "AAPL"
BENCHMARK = "SPY"
START_DATE = "2010-01-04"
END_DATE = None
INITIAL_AMOUNT = 10_000
CEO_DAYS_BEFORE = 60
CEO_DAYS_AFTER = 120

# Data loading, date selection, event filtering, and formatting helpers are imported from analysis_engine.


## Imported Helpers

The notebook intentionally avoids local helper definitions. Loading, date selection, CEO filtering, formatting, and calculations all come from `analysis_engine`.

In [ ]:
# Helpers are imported in the first cell. Keep this section as a reminder that
# notebook code should focus on story assembly and visualization.


In [ ]:
prices = load_prices(TICKER)
benchmark_prices = load_prices(BENCHMARK)
fundamentals = load_fundamentals(TICKER)
events = load_events(TICKER)

end_date = pd.Timestamp(END_DATE) if END_DATE else min(prices["date"].max(), benchmark_prices["date"].max())
start = row_on_or_after(prices, START_DATE)
end = row_on_or_before(prices, end_date)
benchmark_start = row_on_or_after(benchmark_prices, START_DATE)
benchmark_end = row_on_or_before(benchmark_prices, end_date)
years = (end["date"] - start["date"]).days / 365.25

comparison = compare_investments(
    TICKER,
    start["adj_close"],
    end["adj_close"],
    BENCHMARK,
    benchmark_start["adj_close"],
    benchmark_end["adj_close"],
    INITIAL_AMOUNT,
    years=years,
)

stock_path = prices[(prices["date"] >= start["date"]) & (prices["date"] <= end["date"])].copy()
benchmark_path = benchmark_prices[(benchmark_prices["date"] >= benchmark_start["date"]) & (benchmark_prices["date"] <= benchmark_end["date"])].copy()
stock_path["wealth"] = INITIAL_AMOUNT * stock_path["adj_close"] / start["adj_close"]
benchmark_path["wealth"] = INITIAL_AMOUNT * benchmark_path["adj_close"] / benchmark_start["adj_close"]

comparison


## 1. Investment Outcome

First answer the user's main question: what did the investment become, and did it beat the market?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8))
ax.plot(stock_path["date"], stock_path["wealth"], color="#2563eb", linewidth=2.7, label=TICKER)
ax.plot(benchmark_path["date"], benchmark_path["wealth"], color="#52525b", linewidth=2.3, label=BENCHMARK)

for label, result, color in [(TICKER, comparison.first, "#2563eb"), (BENCHMARK, comparison.second, "#52525b")]:
    ax.scatter(end["date"], result.final_value, color=color, s=68, zorder=3)
    ax.annotate(
        f"{label}: {money(result.final_value)}",
        xy=(end["date"], result.final_value),
        xytext=(-142, 26 if label == TICKER else -36),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->", "color": color},
        fontsize=10,
        color=color,
    )

ax.set_title(f"{money(INITIAL_AMOUNT)} in {TICKER} vs {BENCHMARK}", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Portfolio value")
ax.yaxis.set_major_formatter(lambda value, pos: money(value))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(
    f"{TICKER} turned {money(INITIAL_AMOUNT)} into {money(comparison.first.final_value)}, "
    f"versus {money(comparison.second.final_value)} for {BENCHMARK}. "
    f"Winner: {comparison.winner}, ahead by {money(comparison.final_value_difference)}."
)


## 2. Risk Journey

A strong return is only half the story. The drawdown shows what the investor had to endure.

In [ ]:
drawdown = calculate_max_drawdown(stock_path["wealth"].tolist())
stock_path["running_peak"] = stock_path["wealth"].cummax()
stock_path["drawdown"] = stock_path["wealth"] / stock_path["running_peak"] - 1

peak_date = stock_path.iloc[drawdown.peak_index]["date"].date().isoformat()
trough_date = stock_path.iloc[drawdown.trough_index]["date"].date().isoformat()
recovery_date = None if drawdown.recovery_index is None else stock_path.iloc[drawdown.recovery_index]["date"].date().isoformat()

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.fill_between(stock_path["date"], stock_path["drawdown"] * 100, 0, color="#fee2e2")
ax.plot(stock_path["date"], stock_path["drawdown"] * 100, color="#dc2626", linewidth=2)
trough = stock_path.iloc[drawdown.trough_index]
ax.scatter(trough["date"], trough["drawdown"] * 100, color="#991b1b", s=72, zorder=3)
ax.annotate(
    f"Max drawdown: {pct(drawdown.max_drawdown)}\n{peak_date} to {trough_date}",
    xy=(trough["date"], trough["drawdown"] * 100),
    xytext=(24, -10),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#991b1b"},
    fontsize=10,
    color="#7f1d1d",
)
ax.set_title(f"The hardest part of holding {TICKER}", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Drawdown from prior high")
ax.yaxis.set_major_formatter(lambda value, pos: f"{value:.0f}%")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"The deepest decline was {pct(drawdown.max_drawdown)}. Recovery date: {recovery_date or 'not recovered in window'}.")


## 3. Business Quality

Now connect the stock chart to the business: revenue growth, profitability, and balance-sheet pressure.

In [ ]:
income = annual_statement_rows(fundamentals, "income_statement")
balance = annual_statement_rows(fundamentals, "balance_sheet")
latest_income = income.iloc[-1]
previous_income = income.iloc[-2]
latest_balance = balance.iloc[-1]

revenue_growth = calculate_revenue_growth(latest_income["total_revenue"], previous_income["total_revenue"])
gross_margin = calculate_gross_margin(latest_income["gross_profit"], latest_income["total_revenue"])
net_margin = calculate_net_margin(latest_income["net_income"], latest_income["total_revenue"])
debt_to_cash = calculate_debt_to_cash(latest_balance["long_term_debt"], latest_balance["cash_and_cash_equivalents"])

recent_income = income.tail(5).copy()
recent_income["year"] = pd.to_datetime(recent_income["period_end"]).dt.year

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.bar(recent_income["year"], recent_income["total_revenue"] / 1e9, color="#2563eb", alpha=0.85, label="Revenue")
ax.plot(recent_income["year"], recent_income["net_income"] / 1e9, color="#0f766e", marker="o", linewidth=2.5, label="Net income")
ax.set_title(f"{TICKER} business trend", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Billions USD")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

pd.DataFrame([
    {"metric": "Revenue growth", "value": pct(revenue_growth), "label": classify_growth_rate(revenue_growth)},
    {"metric": "Gross margin", "value": pct(gross_margin), "label": classify_margin(gross_margin)},
    {"metric": "Net margin", "value": pct(net_margin), "label": classify_margin(net_margin)},
    {"metric": "Debt to cash", "value": f"{debt_to_cash:,.2f}x", "label": "balance sheet pressure"},
])


## 4. Simple Valuation

Keep this first pass basic: latest price divided by latest annual diluted EPS.

In [ ]:
latest_price = float(end["adj_close"])
latest_eps = float(latest_income["diluted_eps"])
pe_ratio = calculate_pe_ratio(latest_price, latest_eps)
pe_label = classify_pe_level(pe_ratio)
market_cap = fundamentals["company"].get("marketCap")

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(["P/E"], [pe_ratio], color="#7c3aed", width=0.42)
ax.axhline(25, color="#71717a", linestyle="--", linewidth=1, label="25x reference")
ax.text("P/E", pe_ratio, f"{pe_ratio:,.1f}x\n{pe_label}", ha="center", va="bottom", fontsize=12)
ax.set_title(f"Simple valuation snapshot for {TICKER}", loc="left", fontsize=15, fontweight="bold")
ax.set_ylabel("Multiple")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Latest market cap in the data: {compact_money(market_cap)}. Latest annual EPS: {latest_eps:,.2f}. Simple P/E: {pe_ratio:,.1f}x, classified as {pe_label}.")


## 5. CEO Events As Context

CEO events are one chapter of the broader story. We show where they happened, then zoom into the market reaction around each unique event date.

In [ ]:
story_events = ceo_events(events, start["date"], end["date"])

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(stock_path["date"], stock_path["wealth"], color="#2563eb", linewidth=2.3, alpha=0.72)
for _, event in story_events.iterrows():
    event_row = row_on_or_after(stock_path, event["date"])
    ax.axvline(event_row["date"], color="#d97706", linewidth=1, alpha=0.45)
    ax.scatter(event_row["date"], event_row["wealth"], color="#d97706", s=58, zorder=3)
    ax.annotate(
        f"{event['executive_name']}\n{event['action']}",
        xy=(event_row["date"], event_row["wealth"]),
        xytext=(16, 28),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->", "color": "#b45309"},
        fontsize=10,
        color="#92400e",
    )
ax.set_title(f"CEO milestones inside the full {TICKER} story", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Value of initial investment")
ax.yaxis.set_major_formatter(lambda value, pos: money(value))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

story_events


In [ ]:
unique_events = story_events.drop_duplicates(subset=["date"]).reset_index(drop=True)
price_values = prices["adj_close"].tolist()
reaction_rows = []

if unique_events.empty:
    print("No CEO event found inside this selected period.")
else:
    fig, axes = plt.subplots(len(unique_events), 1, figsize=(11, 4.0 * len(unique_events)), squeeze=False)
    for ax, (_, event) in zip(axes.flatten(), unique_events.iterrows()):
        event_index = int(row_on_or_after(prices, event["date"]).name)
        reaction = calculate_event_return(price_values, event_index, CEO_DAYS_BEFORE, CEO_DAYS_AFTER)
        left = max(0, event_index - CEO_DAYS_BEFORE)
        right = min(len(prices) - 1, event_index + CEO_DAYS_AFTER)
        window = prices.iloc[left : right + 1].copy()
        event_price = prices.iloc[event_index]["adj_close"]
        window["day"] = window.index - event_index
        window["relative_return"] = window["adj_close"] / event_price - 1

        ax.plot(window["day"], window["relative_return"] * 100, color="#2563eb", linewidth=2.2)
        ax.axvline(0, color="#b45309", linewidth=2)
        ax.axhline(0, color="#71717a", linewidth=1)
        ax.set_title(f"Reaction around {event['executive_name']} {event['action']}", loc="left", fontsize=14, fontweight="bold")
        ax.set_xlabel("Trading days from CEO event")
        ax.set_ylabel("Return vs event day")
        ax.yaxis.set_major_formatter(lambda value, pos: f"{value:.0f}%")
        ax.spines[["top", "right"]].set_visible(False)

        reaction_rows.append({
            "date": event["date"].date().isoformat(),
            "executive": event["executive_name"],
            "action": event["action"],
            "before_to_event": reaction.before_to_event_return,
            "event_to_after": reaction.event_to_after_return,
        })

    plt.tight_layout()
    plt.show()

pd.DataFrame(reaction_rows)


## Final Story Draft

This is the shape the frontend can eventually render as narrative copy around the charts.

In [ ]:
print(
    f"A {money(INITIAL_AMOUNT)} investment in {TICKER} from {start['date'].date().isoformat()} to {end['date'].date().isoformat()} "
    f"became {money(comparison.first.final_value)}, compared with {money(comparison.second.final_value)} for {BENCHMARK}. "
    f"The business most recently showed {pct(revenue_growth)} annual revenue growth and a {pct(net_margin)} net margin. "
    f"The simple P/E based on latest annual EPS is {pe_ratio:,.1f}x, which this first-pass classifier calls {pe_label}. "
    f"The hardest holding period was a {pct(drawdown.max_drawdown)} drawdown, with recovery on {recovery_date or 'no recovery inside the window'}. "
    "CEO events are shown as context, then zoomed separately so we do not imply causality from a single chart."
)
